## Tools

- Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
    1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
    2. A function or coroutine to execute.

In [2]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

c:\Users\mani2\Documents\RAG-Mastery\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AIMessage(content='<think>\nOkay, so I need to figure out why parrots talk. Let me start by recalling what I know about parrots. They\'re birds, right? Some species like the African Grey or the Macaw are known for their ability to mimic human speech. But why do they do that?\n\nFirst, maybe it\'s a form of communication. In the wild, parrots use vocalizations to interact with each other. They might mimic sounds to bond with their flock members or to identify each other. So in captivity, when a parrot is around humans, maybe they start mimicking human speech because they\'re trying to communicate with their human companions.\n\nAnother thought: parrots are social animals. They form strong bonds with their flock or human caregivers. If they\'re in an environment where they\'re isolated, they might talk more to seek attention or comfort. So talking could be a way to keep their human "flock" engaged or to avoid being left alone.\n\nI\'ve heard that some parrots talk because they\'re traine

In [3]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools=model.bind_tools([get_weather])

In [4]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)

for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Boston. I need to use the get_weather function. The function requires a location parameter. Boston is the location here. So I should call get_weather with location set to "Boston". Let me make sure there\'s no other parameters needed. The required field is just location. Alright, that\'s all. I\'ll format the tool call correctly.\n', 'tool_calls': [{'id': '2w087wrek', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 103, 'prompt_tokens': 154, 'total_tokens': 257, 'completion_time': 0.160679252, 'completion_tokens_details': {'reasoning_tokens': 79}, 'prompt_time': 0.005938239, 'prompt_tokens_details': None, 'queue_time': 0.046321261, 'total_time': 0.166617491}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls'

#### Tool Execution Loops

In [5]:
# Step 1: Model generates tool calls
messages = [{
    "role": "user", 
    "content": "What's the weather in Boston?"
    }]
    
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

In [6]:
# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

In [7]:
# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The weather in Boston is sunny. Let me know if you need more details! ☀️


In [8]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. I need to use the get_weather function. Let me check the function parameters. The required parameter is location, which should be a string. Boston is the location here. So I\'ll call get_weather with location set to "Boston". Make sure the JSON is correctly formatted with the name and arguments. No other parameters are needed. Just pass "Boston" as the location.\n', 'tool_calls': [{'id': '8zend0jff', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 111, 'prompt_tokens': 153, 'total_tokens': 264, 'completion_time': 0.165772164, 'completion_tokens_details': {'reasoning_tokens': 87}, 'prompt_time': 0.006337724, 'prompt_tokens_details': None, 'queue_time': 0.048373325, 'total_time': 0.172109888}, 'model_name': 